# Notebook 06 – Persona Overrides

Discover UI adaptations associated with each shopping persona.

This notebook does **not** build complete interfaces. It only identifies UI elements that:
1. Differ from the default UI (Notebook 05)
2. Are supported by multi-source evidence (Statistics, Random Forest, SHAP, Association Rules)

Default values are never duplicated.

## Inputs
- `data/processed/clean_dataset.csv`
- `data/outputs/global_defaults.json`
- `data/outputs/desktop_defaults.json`
- `data/outputs/mobile_defaults.json`
- Notebook 03 statistical results
- Notebook 04 Random Forest + SHAP
- Association rules (persona-only rules, when available)

## Outputs
- `data/outputs/persona_overrides.json`
- `reports/PersonaOverrides/persona_overrides.xlsx`

In [8]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.persona_overrides.repository import run_persona_override_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("PersonaOverrides")
OUTPUT_DIR = PATHS.data_outputs

print(f"Reports: {REPORTS}")
print(f"JSON output: {OUTPUT_DIR}")

Reports: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonaOverrides
JSON output: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs


## Check Inputs

In [9]:
INPUTS = {
    "Clean dataset": PATHS.data_processed / "clean_dataset.csv",
    "Global defaults": OUTPUT_DIR / "global_defaults.json",
    "Desktop defaults": OUTPUT_DIR / "desktop_defaults.json",
    "Mobile defaults": OUTPUT_DIR / "mobile_defaults.json",
    "Statistical results (NB03)": PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.xlsx",
    "Feature importance (NB04)": PATHS.reports / "Feature_Importance" / "feature_importance.xlsx",
    "SHAP summary (NB04)": PATHS.reports / "Feature_Importance" / "shap_summary.csv",
    "Association rules": PATHS.reports / "AssociationRules" / "base_candidate_rules.csv",
}

for label, path in INPUTS.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'}")

Clean dataset: OK
Global defaults: OK
Desktop defaults: OK
Mobile defaults: OK
Statistical results (NB03): OK
Feature importance (NB04): OK
SHAP summary (NB04): OK
Association rules: OK


## Discover Evidence-Backed Overrides

For each persona and UI element:
- Compare persona majority vs default
- Keep only differences
- Require ≥ 2 evidence sources among Statistics / Random Forest / SHAP / Association Rules

In [10]:
result = run_persona_override_pipeline(PROJECT_ROOT, OUTPUT_DIR, REPORTS)

repositories = result.repositories
table = result.table
summary = result.summary

display(table.head(20) if not table.empty else table)

INFO: Persona overrides: 40 total across 6 personas (avg confidence=0.376)


,Persona,UI_Element,Override_Value,Default_Value,Persona_Count,Persona_Share,Confidence,Support,Evidence,Evidence_Count
0,Browser,desktop_review_display,All Paginated (Show all reviews with pagination),Recent 3- 5 (Show a few recent reviews),18,0.5000,0.500000,0.080357,"Statistics, Random Forest",2
1,Browser,desktop_navigation,Mega Menu (Dropdown with images and subcategor...,Top Bar (Categories always visible across the ...,12,0.3333,0.333333,0.053571,"Random Forest, SHAP",2
2,Browser,desktop_price_display,Strike-through Original (Shows original price ...,With Savings Highlighted (Shows discount amoun...,11,0.3056,0.305556,0.049107,"Statistics, Random Forest, SHAP",3
3,Browser,color_theme_pref,"Vibrant Bold (Bright, energetic colors - excit...","Minimalist Black & White (Clean, high contrast...",10,0.2778,0.277778,0.044643,"Statistics, Random Forest, SHAP",3
4,Browser,urgency_pref,None (I find these annoying and manipulative),Both (Show both stock and time based urgency),10,0.2778,0.277778,0.044643,"Statistics, Random Forest, SHAP",3
5,Browser,form_field_style,Outlined (Border around the field),Rounded Border Fields (Outlined field with rou...,10,0.2778,0.277778,0.044643,"Statistics, Random Forest, SHAP",3
6,Browser,desktop_grid_pref,"2 Columns (Large product images, fewer items p...","3 Columns (Balanced view, moderate item size)",10,0.2778,0.277778,0.044643,"Statistics, Random Forest, SHAP",3
7,Deal Hunter,desktop_image_text_ratio,"Balanced (50% images, 50% text - equal emphasis)","Image-focused (80% images, 20% text - highly v...",17,0.5312,0.531250,0.075893,"Statistics, Random Forest, SHAP",3
8,Deal Hunter,font_style_pref,"3. Rounded Friendly (Soft, approachable fonts ...",2. Classic Serif (Traditional fonts with decor...,16,0.5000,0.500000,0.071429,"Statistics, SHAP",2
9,Deal Hunter,desktop_filter_location,Top Bar (Filters displayed across the top),Sidebar Left (Permanent filter panel on the le...,14,0.4375,0.437500,0.062500,"Random Forest, SHAP",2


## Sample Override JSON

In [11]:
sample = next((item for item in repositories if item["n_overrides"] > 0), repositories[0])
print(json.dumps(sample, indent=2, ensure_ascii=False))

{
  "persona": "Browser",
  "n_respondents": 36,
  "n_overrides": 7,
  "overrides": {
    "color_theme_pref": {
      "value": "Vibrant Bold (Bright, energetic colors - exciting and dynamic)",
      "confidence": 0.277778,
      "support": 0.044643,
      "evidence": [
        "Statistics",
        "Random Forest",
        "SHAP"
      ]
    },
    "urgency_pref": {
      "value": "None (I find these annoying and manipulative)",
      "confidence": 0.277778,
      "support": 0.044643,
      "evidence": [
        "Statistics",
        "Random Forest",
        "SHAP"
      ]
    },
    "form_field_style": {
      "value": "Outlined (Border around the field)",
      "confidence": 0.277778,
      "support": 0.044643,
      "evidence": [
        "Statistics",
        "Random Forest",
        "SHAP"
      ]
    },
    "desktop_grid_pref": {
      "value": "2 Columns (Large product images, fewer items per row)",
      "confidence": 0.277778,
      "support": 0.044643,
      "evidence": [
    

## Overrides per Persona

In [12]:
for persona, count in summary["overrides_per_persona"].items():
    print(f"{persona}: {count}")

Browser: 7
Deal Hunter: 8
Impulsive Buyer: 4
Loyal Customer: 10
Minimalist: 5
Researcher: 6


## Exports

In [13]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")

Export locations:
- persona_overrides_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/persona_overrides.json
- persona_overrides_xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonaOverrides/persona_overrides.xlsx
- persona_overrides_csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonaOverrides/persona_overrides.csv
- summary_md: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/PersonaOverrides/persona_overrides_summary.md


## Final Output

In [14]:
print("Overrides per persona")
for persona, count in summary["overrides_per_persona"].items():
    print(f"  {persona}: {count}")
print(f"Average confidence: {summary['average_confidence']:.4f}")
print("Repository generated.")

Overrides per persona
  Browser: 7
  Deal Hunter: 8
  Impulsive Buyer: 4
  Loyal Customer: 10
  Minimalist: 5
  Researcher: 6
Average confidence: 0.3759
Repository generated.
